# Interactive Dashboards with Filters and Slicers



**Aim:** Create a retail dashboard with region, category, and date filters. In Power BI use slicers; in Tableau use Show Filter or dashboard filter actions. The notebook demonstrates interactive filtering with ipywidgets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
np.random.seed(42)
n = 500
dates = pd.date_range("2024-01-01", periods=180, freq="D")
sales = pd.DataFrame({
    "Order Date": np.random.choice(dates, n),
    "Region": np.random.choice(["North","South","East","West"], n),
    "Category": np.random.choice(["Furniture","Office Supplies","Technology"], n),
    "Sub-Category": np.random.choice(["Chairs","Phones","Binders","Tables","Accessories"], n),
    "Sales": np.round(np.random.gamma(4,120,n),2),
    "Profit": np.round(np.random.normal(35,55,n),2),
    "Quantity": np.random.randint(1,10,n)
}).sort_values("Order Date").reset_index(drop=True)
display(sales.head())

In [ ]:
import plotly.express as px
agg = sales.groupby(["Region","Category"], as_index=False)["Sales"].sum()
px.bar(agg, x="Region", y="Sales", color="Category", barmode="group",
       title="Sales by Region and Category", hover_data={"Sales":":,.2f"}).show()

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    region = widgets.Dropdown(options=["All"]+sorted(sales.Region.unique()), description="Region:")
    category = widgets.Dropdown(options=["All"]+sorted(sales.Category.unique()), description="Category:")
    dates = sorted(sales["Order Date"].dt.date.unique())
    date_range = widgets.SelectionRangeSlider(options=dates, index=(0,len(dates)-1),
        description="Date:", layout=widgets.Layout(width="90%"))
    out = widgets.Output()
    def refresh(*_):
        with out:
            clear_output(wait=True)
            d = sales.copy()
            if region.value != "All": d = d[d.Region == region.value]
            if category.value != "All": d = d[d.Category == category.value]
            d = d[d["Order Date"].dt.date.between(*date_range.value)]
            print("Rows:",len(d)," | Total sales:",round(d.Sales.sum(),2))
            display(d.head())
            if len(d):
                d.groupby("Category").Sales.sum().plot(kind="bar", title="Filtered Sales")
                plt.tight_layout(); plt.show()
    for w in (region,category,date_range): w.observe(refresh,names="value")
    display(widgets.VBox([region,category,date_range,out])); refresh()
except ImportError:
    print("Install ipywidgets with: pip install ipywidgets")

**Observations:** The controls filter records by region, category, and date range. A dashboard should connect the slicers to all relevant charts and KPI cards.\n\n**Result:** An interactive retail dashboard workflow was demonstrated.